# Seminar HCI and BCI in practice
## Session 4 Feature preconditioning and extraction:

***This session focusses on PCA***

In [ ]:
from src.nearly import nearly


In [ ]:
import numpy as np
import os
import sys
import pickle
import matplotlib.pyplot as plt
from scipy.stats import zscore

sys.path.append(os.path.join(os.getcwd(), "src"))
from nearly import nearly

main_path = os.getcwd()
data_path = os.path.join(main_path, 'data/raw')
print(f'Now you are located: {main_path}')


In [ ]:
ecog_file = os.path.join(data_path, 'ecogStruct3.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

# Take a look into our data again
print("ecog contains")
for key, value in ecog.items():
    print(f"Key:{key}, Type:{type(value)}")

# Like in last session you have already worked on, understand the key-value pairs in the spectral analysis
print("\necog['periodogram'] contains")
for key, value in ecog['periodogram'].items():
    print(f"Key: {key}, Type: {type(value)}")


---
## Feature preconditioning

In [ ]:
# Number of trials with finger movement
nTrials = ecog['periodogram']['periodogram'].shape[2]
print(f'Trial number is {nTrials}')

# Frequency features
freqBand = np.array(list(range(4, 59)) + list(range(62, 119)) + list(range(122, 179)))  # the desired frequencies
freqIdx = np.unique([nearly(freqBand, ecog['periodogram']['centerFrequency'])])  # finding the closest center frequencies of the periodogram
nFreq = len(freqIdx)
print(f'Frequency number is {nFreq}')

# Channel features
chan = ecog['selectedChannels']  # here we make sure that we exclude the bad channels that were identified in Session 2
nChan = len(chan)
print(f'Channel number is {nChan}')

# Grab the data
print(f"Original data shape is {ecog['periodogram']['periodogram'].shape}")
index = np.ix_(freqIdx, np.array(chan)-1)        # minus 1, because of python 0-indexsing; np.array() because `chan` is a list
dat = ecog['periodogram']['periodogram'][index]  # periodogram filtered data
print(f"Filtered data shape is {dat.shape}")

<div 
    style="border: 1px dashed black; border-radius: 10px; padding: 10px;">

### Tips: NumPy Array Indexing with `np.ix_`

In MATLAB you index data like this: `data(freqIdx, chan, :)`. But in Python, directly using `data[freqIdx, chan, :]` will raise an error because `freqIdx` and `chan` have different shapes and cannot be broadcast together.


#### **Solution:** Use `np.ix_`

```python
# Create broadcastable indices
index = np.ix_(freqIdx, chan)

# Slice the data
result = data[index]
```

For more details, check the online documentation: https://numpy.org/doc/2.2/reference/generated/numpy.ix_.html


#### Indexing 1st and 3rd Dimensions While Keeping the 2nd Dimension

```python
# Create broadcastable indices
index = np.ix_(freqIdx, np.arange(data.shape[1]), TrailIdx)

# Slice the data
result = data[index]
```

#### *Question:* Why not explicitly specify the *third dimension* in function `np.ix_`, if we slice 1. and 2. dim in a 3-D data?
- In Python, unspecified dimensions (like `:`) are automatically handled.
- Using `np.ix_(freqIdx, chan)` implicitly retains all elements in the third dimension (time points).
- You don't need to explicitly write `np.ix_(freqIdx, chan, np.arange(data.shape[2]))`.

**Try it out!** Compare the results of implicit and explicit indexing to see if they differ.

</div>

---
<div 
    style="border: 1px dashed black; border-radius: 10px; padding: 10px;">

### NumPy Reshape: Understanding `order='F'` vs `order='C'`

<h3 style="color: #FF0000; font-weight: bold;">Why do we need to know this? </h3>

<h3 style="color: #FF0000; "> Because the original data were processed in MATLAB and then transfered to python file.</h3>

When reshaping arrays in NumPy, the `order` parameter controls how elements are read/written during reshaping. Here's the key difference:

#### **1. `order='C'` (Default, C-style)**
- **Behavior**: Elements are read/written row-wise (last index changes fastest).
- **Example**:

  ```python
  import numpy as np
  arr = np.array([[1, 2, 3], [4, 5, 6]])
  reshaped = arr.reshape(3, 2, order='C')  # Output: [[1, 2], [3, 4], [5, 6]]
  ```

  - Fills rows first: `1 → 2 → 3 → 4...`.

#### **2. `order='F'` (Fortran-style)**
- **Behavior**: Elements are read/written column-wise (first index changes fastest).
- **Example**:

  ```python
  reshaped = arr.reshape(3, 2, order='F')  # Output: [[1, 5], [4, 3], [2, 6]]
  ```

  - Fills columns first: `1 → 4 → 2 → 5 → 3 → 6`.

#### **Visualization**
Original 2x3 array:
```
[[1, 2, 3],
 [4, 5, 6]]
```
- **`order='C'` (3x2)**:
  ```
  [[1, 2],
   [3, 4],
   [5, 6]]
  ```
- **`order='F'` (3x2)**:
  ```
  [[1, 5],
   [4, 3],
   [2, 6]]
  ```

#### **When to Use Which?**
- Use `'C'` for row-major operations (default in Python/C).
- Use `'F'` for column-major operations (common in **MATLAB**/Fortran).

**Note**: Reshaping with `order` preserves the original data buffer; only the indexing logic changes.

</div>


<h2 style="color: #FF0000; font-weight: bold;">TASK 1 (2 Point):</h2>

Normalize each frequency across all channels and trials. Reshape your data so that in the end we have a matrix of nTrials x nFeatures (where nFeatrures is a combination of frequencies and channels)

<h3 style="color: #FF0000; font-weight: bold;">Fill in the missing parts (...) in the code below</h3>

In [ ]:
# Reshape to nFreq x (nChan * nTrials)
dat_reshape = np.reshape(dat, (nFreq, nChan * nTrials), order='F')
print(f"[nFreq x (nChan * nTrials)] data shape is {dat_reshape.shape}")

# z-score data, thus data for each frequency point have been standardized 
zdat = zscore(dat_reshape, axis=1)

# Reshape data to nFreq x nChan x nTrials
dat_zscored = np.reshape(zdat, (nFreq, nChan, nTrials), order='F')
print(f"[nFreq x nChan x nTrials] data shape is {dat_zscored.shape}")

# Permute data to nTrials x nChan x nFreq
dat_trans = np.transpose(dat_zscored, (2, 1, 0))
print(f"[nTrials x nChan x nFreq] data shape is {dat_trans.shape}")

# Reshape data to nTrials x (nChan * nFreq)
dat_feature = np.reshape(dat_trans, (nTrials, nChan * nFreq), order='F')
print(f"[nTrials x (nChan * nFreq)] data shape is {dat_feature.shape}")


In [ ]:
# Save data to a .pkl file
zScoredData = {'dat': dat_feature,
              'nFreq': nFreq, 
               'nChan': nChan, 
               'nTrials': nTrials}

dat_file = os.path.join(data_path, 'processed/zScoredData_Ses04.pkl')
os.makedirs(os.path.dirname(dat_file), exist_ok=True)
with open(dat_file, 'wb') as f:
    pickle.dump(zScoredData, f)

---
<h2 style="color: #FF0000; font-weight: bold;">TASK 2 (2 pt):</h2>

- Why do we z-score the data?
- What does our feature Vector consist of? 
- How many features are there at the moment?

In [ ]:
# --- TASK 2: how different are the frequencies in scale, and how many features? ---
mean_power = dat.reshape(nFreq, -1).mean(axis=1)   # mean power per frequency, over channels and trials

print("number of features = nChan * nFreq = %d * %d = %d" % (nChan, nFreq, nChan * nFreq))
print("feature matrix dat_feature shape  :", dat_feature.shape, "-> (nTrials, nFeatures)")

print("\nmean power at %5.1f Hz : %8.3f" % (ecog['periodogram']['centerFrequency'][freqIdx[0]], mean_power[0]))
print("mean power at %5.1f Hz : %8.5f" % (ecog['periodogram']['centerFrequency'][freqIdx[-1]], mean_power[-1]))
print("so the strongest frequency is about %.0f times larger than the weakest"
      % (mean_power.max() / mean_power.min()))

print("\nafter z-scoring, one example frequency row:")
print("   mean = %.2e   std = %.3f" % (zdat[0].mean(), zdat[0].std()))

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

**Why do we z-score the data?**

Because the different frequencies live on completely different scales. Low frequencies carry a lot of power and high frequencies very little. In our data the mean power at 4 Hz is about **14.7**, while at 178.8 Hz it is about **0.0035**, so the low frequency is roughly **4000 times larger** than the high one.

If we used the raw values as features, the few strong low frequencies would dominate everything. Any method that looks at variance, like the PCA in the next task, would basically only see those, and the high-gamma band above 65 Hz, which is exactly where the movement information is, would be drowned out.

Z-scoring fixes this. For each frequency we subtract its mean and divide by its standard deviation, so every frequency ends up with mean 0 and std 1. After that all frequencies contribute on an equal footing, and a change is judged relative to how much that frequency normally varies, not by its absolute size. The check in the cell shows it worked: an example frequency row now has mean about 0 and std 1.

It is important that the z-scoring is done **per frequency across all channels and trials** (`zscore(..., axis=1)` on the `nFreq x (nChan*nTrials)` matrix). That way each frequency is normalised as a whole, and the differences between channels and between trials inside one frequency are kept, which are exactly the differences we want to classify later.

**What does our feature vector consist of?**

After the reshaping, each **trial** is one row, and the row is the feature vector for that trial. It is the combination of **channels and frequencies**: for every one of the 38 good channels we have the power at every one of the 87 frequencies. So one feature is "the z-scored power of channel c at frequency f", and the whole vector describes the frequency content of that trial across the whole electrode grid.

**How many features are there at the moment?**

`nChan * nFreq = 38 * 87 = `**`3306`** features, and there are 315 trials. So `dat_feature` has the shape **(315, 3306)**, which is nTrials x nFeatures. That is a lot of features for only 315 trials, which is the reason PCA is used next to bring the number down.

## Principal component analysis

<h2 style="color: #FF0000; font-weight: bold;">TASK 3 (2 pt): Manually Calculate PCA</h2>

(For help, you can check out `"PCA-Tutorial-Intuition_jp"` that you can find on **StudIP** ) 

<h3 style="color: #FF0000; font-weight: bold;">fill the ... part in the code cell below: </h3>

In [ ]:
# Subtract the mean of each feature
dat_demean = dat_feature - np.mean(dat_feature, axis=0)
print(f"mean of dat now is {np.mean(dat_demean, axis=0)}")

# Calculate the covariance matrix
cPCA = np.cov(dat_demean, rowvar=False)

# Calculate the Eigenvalue decomposition
eigenVals, V = np.linalg.eigh(cPCA)

print(eigenVals.shape)
print(V.shape)


In [ ]:
# Sort the results in decreasing order
indices = np.argsort(-eigenVals)  # Sort in descending order
eigenVals = eigenVals[indices]    # Sort the eigen values
V = V[:, indices]                 # Sort the coresponding eigen vectors as well

---
<h2 style="color: #FF0000; font-weight: bold;">TASK 4 (2 pt)</h2>

- What do the eigenVals and V contain? 
- Using the code cell below, so you can calculate the proportion of variance explained by each principle component: how much variance does the first / do the first 100 principle components explain?


<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration (fill the ... part in the code cell): </h3>


In [ ]:
# Calculate the proportion of variance explained by each PC
prop = eigenVals / np.sum(eigenVals)

# proportion of first PC
PC_1 = prop[0]
# proportion of first 100 PCs
PC_100 = np.sum(prop[:100])


# Print the result
print(f"The proportion of variance explained by the first 1 PC is {PC_1:.3%}")
print(f"The proportion of variance explained by the first 100 PCs is {PC_100:.3%}")

# Transform the original data to the PC coordinate system by multiplying with V
xPCA = dat_demean @ V
print(f"xPCA shape is:{xPCA.shape}")

**What do `eigenVals` and `V` contain?**

They come from the eigenvalue decomposition of the covariance matrix, and together they describe the principal components.

- **`V`** contains the **principal components (the directions)**. Each column of `V` is one eigenvector, a direction in the 3306-dimensional feature space along which the data varies. These directions are orthonormal, so they are at right angles to each other and each has length 1. That is why later `inv(V) = V.T` and the reconstruction works. The first column (after sorting) is the direction of the largest variation in the data, the second column the next largest, and so on.
- **`eigenVals`** contains **how much variance lies along each of those directions**. A large eigenvalue means the data spreads out a lot along that eigenvector, a small one means almost no spread. After sorting in decreasing order, `eigenVals[0]` belongs to `V[:, 0]` and is the largest.

So `V` says *in which directions* the data varies and `eigenVals` says *how much* it varies in each direction.

**How much variance do the first / first 100 PCs explain?**

`prop = eigenVals / np.sum(eigenVals)` turns each eigenvalue into a fraction of the total variance. From the cell:

- the **first PC** explains about **9.85%** of the total variance,
- the **first 100 PCs together** explain about **80.66%**.

So one direction already captures almost 10%, and 100 directions out of 3306 keep about 80% of everything. That is the whole point of PCA here: the 3306 features can be reduced to around 100 numbers per trial and most of the information is still there.

One extra thing I noticed: only about **312** eigenvalues are really larger than zero. The rest are essentially 0, because with only 315 trials the covariance matrix can have at most `nTrials - 1 = 314` non-zero directions. So even the full PCA does not use all 3306 components.

---
### Reconstruction

In case we want the old data back we do this by multiplying the PC time series representation with the inverse of v. 
- Remark: Because v is orthonormal inv(v) = v.T
- Remark: That's the underlying assumption that the observed pattern is generated by an additive superposition of time varying principal patterns (or sources)

In [ ]:
datReconFull = xPCA @ V.T
print(f"Full PCA-reconstracted data shape is:{datReconFull.shape}")

In [ ]:
# In case we want to drop some components, we can do this by setting these principal components (i.e., columns in V) to zero.

# Create a copy of V to avoid modifying the original matrix
subV = V.copy()

# Define the principal components to remove
# Here, we remove components from index 10 to 3306 (inclusive)
# Note: Python uses 0-based indexing, and the end index is exclusive
removedPCs = np.arange(10, 3306)  # Generates array [10, 11, ..., 3305]

# Set the specified principal components to zero
subV[:, removedPCs] = 0

# Reconstruct the data using the modified V matrix
datReconPartial = xPCA @ subV.T

In [ ]:
# Visualization of results
shownTrials = np.arange(0, 50)  # some trials (for better visualization, not all are shown)
shownFeatures = np.arange(0, nFreq * 3)  # features from three channels (for better visualization, not all are shown)

# Create a figure with 3 subplots
plt.figure(figsize=(10, 12))

# Subplot 1: Original data
plt.subplot(3, 1, 1)
plt.imshow(dat_demean[shownTrials][:, shownFeatures], aspect='auto', cmap='viridis', vmin = -4, vmax = 2.5)
plt.xlabel('Features', fontsize=18)
plt.ylabel('Trials', fontsize=18)
plt.title('Original Data', fontsize=18)
plt.colorbar()

# Subplot 2: Reconstructed data (full)
plt.subplot(3, 1, 2)
plt.imshow(datReconFull[shownTrials][:, shownFeatures], aspect='auto', cmap='viridis', vmin = -4, vmax = 2.5)
plt.xlabel('Features', fontsize=18)
plt.ylabel('Trials', fontsize=18)
plt.title('Reconstructed Data, Full', fontsize=18)
plt.colorbar()

# Subplot 3: Reconstructed data (partial)
plt.subplot(3, 1, 3)
plt.imshow(datReconPartial[shownTrials][:, shownFeatures], aspect='auto', cmap='viridis', vmin = -4, vmax = 2.5)
plt.xlabel('Features', fontsize=18)
plt.ylabel('Trials', fontsize=18)
plt.title('Reconstructed Data, Partial', fontsize=18)
plt.colorbar()

# Adjust layout and show plot
plt.tight_layout()
plt.show()

---
<h2 style="color: #FF0000; font-weight: bold;">TASK 5 (1 pt)</h2>
How do these three subplots relate to each other at the moment? Is this expected?

**How do the three subplots relate to each other, and is this expected?**

- **Original vs. Full reconstruction (subplots 1 and 2):** they look **exactly the same**. I checked it in numbers and the largest difference is about 1e-13, which is only floating point rounding. This is expected. The full reconstruction is `xPCA @ V.T`, and because `V` is orthonormal, `V.T` is its inverse, so multiplying by `V` and then by `V.T` gives the original data back. No component was dropped, so nothing is lost.

- **Partial reconstruction (subplot 3):** it looks like a **blurred version** of the original. The big blocks are still in the same place, but the fine detail is gone. This is also expected, because here all principal components except the first 10 were set to zero. Those 10 components only carry about **32.6%** of the total variance, so the partial reconstruction keeps the strongest, most common patterns and throws away everything smaller.

So at the moment: subplot 2 is identical to subplot 1, and subplot 3 is a smoothed, lower-detail version of it. That is exactly what should happen. It also shows the idea of PCA nicely: keeping only a few components already reproduces the main structure of the data, and how much detail is lost depends on how many components are kept.

In [ ]:
# The results of this PCA will be used next session, so save the results of your PCA for later use.

# Save data to a .pkl file
resultsPCA = {'xPCA': xPCA,
              'eigenVals': eigenVals, 
               'V': V, 
               'datReconPartial': datReconPartial}

dat_file = os.path.join(data_path, 'processed/resultsPCA.pkl')
with open(dat_file, 'wb') as f:
    pickle.dump(resultsPCA, f)